# Лабораторная работа IV. Классификация тональности отзывов Кинопоиска

**Датасет:** [Kinopoisk Movie Reviews](https://www.kaggle.com/datasets/mikhailklemin/kinopoisks-movies-reviews) (Kaggle, папки `pos` / `neg` / `neu` с `.txt`-отзывами)  
**Задача:** классификация тональности (позитивный / негативный / нейтральный отзыв)  
**Фреймворк:** scikit-learn + CatBoost + Optuna  

В работе построен полный pipeline обработки текста и классификации:
подготовка Kaggle-датасета в `LAB_IV/data/dataset`, сборка рабочей таблицы из `.txt`-файлов,
разведочный анализ с Plotly-графиками и облаками слов,
предобработка (нижний регистр → пунктуация → токенизация → стоп-слова → лемматизация spaCy),
подготовка данных к обучению (train/test split, TF-IDF векторизация),
сравнение трёх обязательных baseline-моделей (LogisticRegression, LinearSVC, CatBoostClassifier),
подбор гиперпараметров через Optuna, анализ ошибок
и тест на пяти новых текстах в разных стилях.


## Оглавление

1. [Подготовка окружения](#Подготовка-окружения)
2. [Загрузка данных](#Загрузка-данных)
3. [Разведочный анализ данных](#Разведочный-анализ-данных)
4. [Предобработка текстов](#Предобработка-текстов)
5. [Подготовка данных к обучению](#Подготовка-данных-к-обучению)
6. [Классификация](#Классификация)
7. [Анализ ошибок](#Анализ-ошибок)
8. [Тест на новых текстах](#Тест-на-новых-текстах)
9. [Выводы](#Выводы)

## Подготовка окружения

Первый блок фиксирует воспроизводимость, подключает все нужные библиотеки,
задаёт режим обучения CPU/CUDA для CatBoost и загружает языковую модель spaCy
для русского языка. Фиксация `SEED=42` обеспечивает одинаковые результаты
при повторном запуске.


### 1.1 Импорты и глобальные настройки

Все импорты собраны в одну ячейку согласно рекомендациям PEP 8.
`warnings.filterwarnings("ignore")` убирает малоинформативные предупреждения
sklearn о сходимости при быстрых экспериментах.


In [ ]:
import json
import os
import re
import random
import shutil
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud

try:
    import spacy
    from spacy.lang.ru.stop_words import STOP_WORDS as SPACY_RU_STOPWORDS
except ImportError as exc:
    raise RuntimeError("spaCy не установлен. Выполните: %pip install -q spacy") from exc

try:
    import optuna
except ImportError as exc:
    raise RuntimeError("Optuna обязательна для подбора гиперпараметров. Выполните: %pip install -q optuna") from exc

try:
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise RuntimeError("CatBoost обязателен для этой работы. Выполните: %pip install -q catboost") from exc

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score,
)

from tqdm import tqdm
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed()
print("Imports OK")


### 1.2 Переключатель CPU/CUDA для обучения

`DEVICE_MODE = "auto"` — безопасный режим по умолчанию: на машине с CUDA
CatBoost будет обучаться на GPU, а на macOS или другом окружении без CUDA
автоматически останется CPU. Для явного запуска измените значение на
`"cpu"` или `"cuda"` и перезапустите ячейки обучения.


In [ ]:
DEVICE_MODE = "auto"  # варианты: "auto", "cpu", "cuda"
CATBOOST_CUDA_DEVICES = "0"  # например: "0" или "0:1" для нескольких GPU


def get_cuda_status() -> tuple[bool, str]:
    try:
        import torch
    except Exception as exc:
        return False, f"torch недоступен: {type(exc).__name__}: {exc}"

    try:
        cuda_available = bool(torch.cuda.is_available())
    except Exception as exc:
        return False, f"torch.cuda.is_available() завершился с ошибкой: {type(exc).__name__}: {exc}"

    torch_cuda_version = getattr(getattr(torch, "version", None), "cuda", None)
    status = f"torch={getattr(torch, '__version__', 'unknown')}, torch CUDA={torch_cuda_version}"
    return cuda_available, status


def is_cuda_available() -> bool:
    cuda_available, _ = get_cuda_status()
    return cuda_available


def resolve_training_device(mode: str = DEVICE_MODE) -> dict:
    normalized_mode = str(mode).strip().lower()
    allowed_modes = {"auto", "cpu", "cuda"}
    if normalized_mode not in allowed_modes:
        raise ValueError(f"DEVICE_MODE должен быть одним из {sorted(allowed_modes)}, получено: {mode!r}")

    cuda_available, cuda_status = get_cuda_status()
    use_cuda = normalized_mode == "cuda" or (normalized_mode == "auto" and cuda_available)

    if normalized_mode == "cuda" and not cuda_available:
        raise RuntimeError(
            "Выбран DEVICE_MODE='cuda', но CUDA недоступна в текущем окружении. "
            "Проверьте CUDA/PyTorch на GPU-машине или переключите DEVICE_MODE на 'auto'/'cpu'. "
            f"Статус: {cuda_status}"
        )

    catboost_task_type = "GPU" if use_cuda else "CPU"
    catboost_params = {"task_type": catboost_task_type}
    if use_cuda:
        catboost_params["devices"] = CATBOOST_CUDA_DEVICES

    return {
        "requested_mode": normalized_mode,
        "cuda_available": cuda_available,
        "cuda_status": cuda_status,
        "catboost_task_type": catboost_task_type,
        "catboost_params": catboost_params,
    }


TRAINING_DEVICE = resolve_training_device()
CATBOOST_TASK_TYPE = TRAINING_DEVICE["catboost_task_type"]
CATBOOST_DEVICE_PARAMS = TRAINING_DEVICE["catboost_params"]

print("Requested device mode:", TRAINING_DEVICE["requested_mode"])
print("CUDA available:", TRAINING_DEVICE["cuda_available"])
print("CUDA status:", TRAINING_DEVICE["cuda_status"])
print("CatBoost task_type:", CATBOOST_TASK_TYPE)
if CATBOOST_TASK_TYPE == "GPU":
    print("CatBoost CUDA devices:", CATBOOST_CUDA_DEVICES)


### 1.3 Функции визуализации Plotly

Все графики строятся через Plotly. Общие настройки и функции вынесены в отдельную раннюю ячейку,
чтобы дальше в ноутбуке переиспользовать готовые вызовы без дублирования оформления.


In [ ]:
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_COLORS = px.colors.qualitative.Set2

BASIC_SW = {
    "и", "в", "на", "что", "это", "как", "с", "из", "а", "но",
    "по", "за", "то", "так", "же", "был", "была", "всё", "он",
    "она", "они", "или", "к", "у", "от", "для", "его", "её",
    "их", "ещё", "уже", "о", "об", "при", "до", "со", "чем",
    "во", "бы", "ли", "тут", "тоже", "всех", "этот", "этого",
}


def show_plotly_figure(fig, title: str | None = None, *, height: int = 480):
    """Apply the shared notebook style and render a Plotly figure."""
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        title=title,
        title_x=0.02,
        height=height,
        margin=dict(l=40, r=30, t=75 if title else 45, b=40),
        legend_title_text="Класс",
        font=dict(size=12),
    )
    fig.show()
    return fig


def make_class_color_map(labels) -> dict:
    labels = [str(label) for label in labels]
    return {
        label: PLOTLY_COLORS[i % len(PLOTLY_COLORS)]
        for i, label in enumerate(labels)
    }


def plot_class_counts_and_lengths(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    word_count_col: str = "word_count",
):
    counts = data[target_col].value_counts()
    color_map = make_class_color_map(counts.index)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Распределение классов", "Распределение длин отзывов"),
    )
    fig.add_trace(
        go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            text=[f"{value:,}" for value in counts.values],
            textposition="outside",
            marker_color=[color_map[str(label)] for label in counts.index],
            name="Количество отзывов",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    for label in counts.index:
        subset = data.loc[data[target_col] == label, word_count_col]
        fig.add_trace(
            go.Histogram(
                x=subset,
                nbinsx=60,
                opacity=0.55,
                marker_color=color_map[str(label)],
                name=str(label),
            ),
            row=1,
            col=2,
        )

    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text="Класс", row=1, col=1)
    fig.update_yaxes(title_text="Количество отзывов", row=1, col=1)
    fig.update_xaxes(title_text="Количество слов", range=[0, 500], row=1, col=2)
    fig.update_yaxes(title_text="Частота", row=1, col=2)
    return show_plotly_figure(fig, "Базовая статистика датасета", height=500)


def plot_wordclouds_by_class(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    text_col: str = "review",
    max_words: int = 150,
):
    classes = data[target_col].dropna().unique().tolist()
    cmaps = ["Blues", "Reds", "Greens", "Purples", "Oranges", "Greys"]
    fig = make_subplots(
        rows=1,
        cols=len(classes),
        subplot_titles=[f"Облако слов — {str(label).capitalize()}" for label in classes],
    )

    for col_idx, label in enumerate(classes, start=1):
        text = " ".join(data.loc[data[target_col] == label, text_col].astype(str))
        wc = WordCloud(
            width=700,
            height=420,
            background_color="white",
            colormap=cmaps[(col_idx - 1) % len(cmaps)],
            max_words=max_words,
            collocations=False,
        ).generate(text)
        fig.add_trace(go.Image(z=wc.to_array(), hoverinfo="skip"), row=1, col=col_idx)
        fig.update_xaxes(visible=False, row=1, col=col_idx)
        fig.update_yaxes(visible=False, row=1, col=col_idx)

    return show_plotly_figure(
        fig,
        "Наиболее частотные слова по классам (до предобработки)",
        height=500,
    )


def get_top_words(texts, *, n: int = 20, stopwords: set[str] | None = None):
    stopwords = BASIC_SW if stopwords is None else stopwords
    words = [
        word.lower()
        for text in texts
        for word in re.findall(r"[а-яёА-ЯЁ]+", str(text))
        if word.lower() not in stopwords and len(word) > 2
    ]
    return Counter(words).most_common(n)


def plot_top_words_by_class(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    text_col: str = "review",
    n: int = 20,
):
    classes = data[target_col].dropna().unique().tolist()
    color_map = make_class_color_map(classes)
    fig = make_subplots(
        rows=1,
        cols=len(classes),
        subplot_titles=[f"Топ-{n} слов — {str(label).capitalize()}" for label in classes],
    )

    for col_idx, label in enumerate(classes, start=1):
        top = get_top_words(data.loc[data[target_col] == label, text_col], n=n)
        if not top:
            continue
        words, freqs = zip(*top)
        fig.add_trace(
            go.Bar(
                x=list(freqs)[::-1],
                y=list(words)[::-1],
                orientation="h",
                marker_color=color_map[str(label)],
                name=str(label),
                showlegend=False,
            ),
            row=1,
            col=col_idx,
        )
        fig.update_xaxes(title_text="Частота", row=1, col=col_idx)

    return show_plotly_figure(fig, f"Топ-{n} слов по классам", height=580)


def plot_confusion_matrix_plotly(
    y_true,
    y_pred,
    labels,
    *,
    title: str = "Матрица ошибок",
):
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    matrix_df = pd.DataFrame(matrix, index=labels, columns=labels)
    fig = px.imshow(
        matrix_df,
        text_auto=True,
        color_continuous_scale="Blues",
        aspect="auto",
        labels=dict(x="Предсказано", y="Истинный класс", color="Количество"),
    )
    fig.update_traces(hovertemplate="Истинный класс: %{y}<br>Предсказано: %{x}<br>Количество: %{z}<extra></extra>")
    return show_plotly_figure(fig, title, height=520)


### 1.4 Загрузка языковой модели spaCy

`ru_core_news_sm` — компактная модель для русского языка.
Используется исключительно для лемматизации:
`disable=["parser", "ner"]` оставляет только `tok2vec` + `morphologizer` + `lemmatizer`,
что ускоряет обработку примерно в 3 раза по сравнению с полным конвейером.

Если модель не установлена, выполните в терминале:
```
python -m spacy download ru_core_news_sm
```


In [ ]:
SPACY_MODEL = "ru_core_news_sm"

try:
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
    print("spaCy model loaded:", nlp.meta["name"])
except OSError as exc:
    raise RuntimeError(
        f"Модель spaCy {SPACY_MODEL!r} не найдена.\n"
        "Установите зависимости в отдельной ячейке и перезапустите kernel:\n"
        "%pip install -q spacy\n"
        f"!python -m spacy download {SPACY_MODEL}"
    ) from exc

Окружение готово. Модель `ru_core_news_sm` загружена без NER и парсера —
нам нужна только лемматизация.


## Загрузка данных

Датасет Kinopoisk Movie Reviews с Kaggle хранится не как один CSV-файл:
в архиве лежат директории классов `pos`, `neg`, `neu`, а каждый отзыв находится
в отдельном `.txt`-файле. В этом разделе приводим такой формат к рабочей таблице
с колонками `review` и `sentiment`.


### 2.1 Подготовка Kaggle-датасета и первичный осмотр

Сначала отдельной ячейкой готовим локальную копию датасета в `LAB_IV/data/dataset`.
Если папки `pos`, `neg`, `neu` уже лежат там, используем их. Если нет, скачиваем
датасет через `kagglehub.dataset_download`, находим внутри скачанной директории
эти папки и копируем их в `LAB_IV/data/dataset`.

После этого собираем кэшированную таблицу `kinopoisk_reviews_prepared.csv`, чтобы
повторные запуски не перечитывали десятки тысяч `.txt`-файлов.


In [ ]:
PREPARED_DATA_FILENAME = "kinopoisk_reviews_prepared.csv"
KAGGLE_DATASET = "mikhailklemin/kinopoisks-movies-reviews"
LABEL_DIR_MAP = {
    "neg": "negative",
    "neu": "neutral",
    "pos": "positive",
}


def find_lab_iv_dir() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if base.name == "LAB_IV" and (base / "data").exists():
            return base
        lab_candidate = base / "LAB_IV"
        if (lab_candidate / "data").exists():
            return lab_candidate
    raise FileNotFoundError("Не найдена директория LAB_IV/data относительно текущего пути запуска")


LAB_IV_DIR = find_lab_iv_dir()
DATA_DIR = LAB_IV_DIR / "data"
DATASET_DIR = DATA_DIR / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATASET_DIR / PREPARED_DATA_FILENAME


def count_txt_files(root: Path) -> int:
    return sum(1 for _ in root.rglob("*.txt")) if root.exists() else 0


def get_review_dir_counts(root: Path) -> dict[str, int]:
    return {
        label_dir: count_txt_files(root / label_dir)
        for label_dir in LABEL_DIR_MAP
    }


def has_review_dirs(root: Path) -> bool:
    return all((root / label_dir).is_dir() for label_dir in LABEL_DIR_MAP)


def find_review_dirs_root(root: Path) -> Path | None:
    root = Path(root)
    if has_review_dirs(root):
        return root
    if root.exists():
        for candidate in root.rglob("*"):
            if candidate.is_dir() and has_review_dirs(candidate):
                return candidate
    return None


def copy_review_dirs(source_root: Path, target_root: Path) -> None:
    source_root = source_root.resolve()
    target_root = target_root.resolve()
    if source_root == target_root:
        return

    target_root.mkdir(parents=True, exist_ok=True)
    for label_dir in LABEL_DIR_MAP:
        src = source_root / label_dir
        dst = target_root / label_dir
        dst.mkdir(parents=True, exist_ok=True)
        for src_file in src.rglob("*.txt"):
            rel_path = src_file.relative_to(src)
            dst_file = dst / rel_path
            dst_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_file, dst_file)


def download_kaggle_dataset() -> Path:
    try:
        import kagglehub
    except ImportError as exc:
        raise RuntimeError(
            "Папки pos/neg/neu не найдены и не установлен kagglehub. "
            "Установите зависимости через uv sync --dev или добавьте kagglehub вручную."
        ) from exc

    errors: list[str] = []
    download_attempts = [
        {"output_dir": str(DATASET_DIR)},
        {},
    ]
    for kwargs in download_attempts:
        try:
            return Path(kagglehub.dataset_download(KAGGLE_DATASET, **kwargs))
        except TypeError as exc:
            errors.append(f"dataset_download({kwargs}) не поддерживается: {exc}")
        except Exception as exc:
            errors.append(f"dataset_download({kwargs}) завершился ошибкой: {exc}")

    raise RuntimeError("Не удалось скачать Kaggle-датасет. Ошибки: " + " | ".join(errors))


def ensure_raw_dataset() -> Path:
    local_root = find_review_dirs_root(DATASET_DIR)
    if local_root is not None:
        copy_review_dirs(local_root, DATASET_DIR)
        return DATASET_DIR

    downloaded_dir = download_kaggle_dataset()
    downloaded_root = find_review_dirs_root(downloaded_dir)
    if downloaded_root is None:
        raise FileNotFoundError(
            "В скачанном Kaggle-датасете не найдены папки pos/neg/neu. "
            f"Проверенный путь: {downloaded_dir.resolve()}"
        )

    copy_review_dirs(downloaded_root, DATASET_DIR)
    return DATASET_DIR


def read_review_text(path: Path) -> str:
    for encoding in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            return path.read_text(encoding=encoding).strip()
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace").strip()


def parse_review_ids(path: Path) -> tuple[int | None, int | None]:
    parts = path.stem.split("-", maxsplit=1)
    if len(parts) != 2:
        return None, None
    try:
        return int(parts[0]), int(parts[1])
    except ValueError:
        return None, None


def build_reviews_dataframe(raw_root: Path) -> pd.DataFrame:
    records = []
    total_files = sum(get_review_dir_counts(raw_root).values())
    progress = tqdm(
        total=total_files,
        desc="Reading review txt files",
        dynamic_ncols=True,
        mininterval=1.0,
        leave=False,
    )

    with progress:
        for label_dir, sentiment in LABEL_DIR_MAP.items():
            label_root = raw_root / label_dir
            for txt_path in sorted(label_root.rglob("*.txt")):
                movie_id, review_id = parse_review_ids(txt_path)
                records.append({
                    "review": read_review_text(txt_path),
                    "sentiment": sentiment,
                    "source_label_dir": label_dir,
                    "source_file": str(txt_path.relative_to(raw_root)),
                    "movie_id": movie_id,
                    "review_id": review_id,
                })
                progress.update(1)

    if not records:
        raise ValueError(f"В {raw_root.resolve()} не найдено ни одного .txt-отзыва")

    result = pd.DataFrame.from_records(records)
    result["movie_id"] = result["movie_id"].astype("Int64")
    result["review_id"] = result["review_id"].astype("Int64")
    return result


raw_dataset_root = ensure_raw_dataset()
raw_counts = get_review_dir_counts(raw_dataset_root)
raw_total = sum(raw_counts.values())

if raw_total == 0:
    raise ValueError(f"Папки pos/neg/neu найдены в {raw_dataset_root.resolve()}, но .txt-файлов в них нет")

cache_is_fresh = False
if DATA_PATH.exists():
    cached_df = pd.read_csv(DATA_PATH)
    has_required_columns = {"review", "sentiment"}.issubset(cached_df.columns)
    cache_is_fresh = has_required_columns and len(cached_df) == raw_total
    if cache_is_fresh:
        df = cached_df
        print("Loaded prepared table:", DATA_PATH.resolve())
    else:
        print("Prepared table exists, but does not match raw dataset. Rebuilding cache...")

if not cache_is_fresh:
    df = build_reviews_dataframe(raw_dataset_root)
    df.to_csv(DATA_PATH, index=False)
    print("Prepared table saved:", DATA_PATH.resolve())

print("Dataset directory:", DATASET_DIR.resolve())
print("Raw review counts:", raw_counts)
print("Prepared table:", DATA_PATH.resolve())
print("Prepared shape:", df.shape)


In [ ]:
print("Prepared data path:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
df.info()


### 2.2 Нормализация столбцов и меток

После сборки из `.txt`-файлов таблица уже содержит `review` и `sentiment`.
Дополнительно нормализуем метки классов, удаляем строки с пустым текстом или
отсутствующей меткой и сохраняем служебные поля `source_file`, `movie_id`,
`review_id` для последующего анализа ошибок.


In [ ]:
required_columns = {"review", "sentiment"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"В подготовленной таблице нет обязательных колонок: {sorted(missing_columns)}")

metadata_columns = [
    column
    for column in ["source_label_dir", "source_file", "movie_id", "review_id"]
    if column in df.columns
]

df = df[["review", "sentiment", *metadata_columns]].copy()

# Normalize sentiment labels to lowercase strings
label_map = {
    "1": "positive", "0": "negative", "-1": "negative",
    "good": "positive", "bad": "negative",
    "pos": "positive", "neg": "negative", "neu": "neutral",
    "positive": "positive", "negative": "negative", "neutral": "neutral",
    "позитивный": "positive", "негативный": "negative", "нейтральный": "neutral",
}

df["sentiment"] = (
    df["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(label_map)
)

# Drop nulls and empty reviews
df.dropna(subset=["review", "sentiment"], inplace=True)
df["review"] = df["review"].astype(str)
df = df[df["review"].str.strip().astype(bool)].reset_index(drop=True)

expected_labels = set(LABEL_DIR_MAP.values())
unexpected_labels = sorted(set(df["sentiment"]) - expected_labels)
if unexpected_labels:
    print("Warning: unexpected labels after normalization:", unexpected_labels)

print("Shape after cleaning:", df.shape)
print("Unique sentiments:", sorted(df["sentiment"].unique()))
print(df["sentiment"].value_counts())


**Вывод после загрузки данных (заполнить после запуска).**

- Сколько `.txt`-отзывов прочитано из папок `pos`, `neg`, `neu`: `...`.
- Размер подготовленной таблицы после нормализации: `...` строк и `...` столбцов.
- Найденные классы: `...`; распределение классов: `...`.
- Пропуски и пустые отзывы после очистки: `...`.
- Краткий комментарий по пригодности данных для классификации тональности: `...`.


## Разведочный анализ данных

Цель EDA — понять распределение классов, характер текстов и наиболее
информативные слова каждого класса. Суммарно не более 8 графиков.


### 3.1 Распределение классов и длины текстов

Два графика в одной строке:
столбчатая диаграмма числа отзывов по классам
и гистограмма длин текстов (в словах) с разбивкой по классам.
Это показывает, есть ли дисбаланс классов
и насколько тексты однородны по длине.


In [ ]:
df["word_count"] = df["review"].str.split().str.len()
plot_class_counts_and_lengths(df)

print(df.groupby("sentiment")["word_count"].describe().round(1))


**Вывод по распределению классов и длинам (заполнить после запуска).**

- Самый частый класс: `...`; самый редкий класс: `...`.
- Степень дисбаланса: `...`; как это может повлиять на `accuracy` и `macro_f1`: `...`.
- Длины отзывов: средняя `...`, медиана `...`, заметные выбросы `...`.
- Решение для обучения: использовать/не использовать `class_weight` и почему: `...`.


### 3.2 Облака слов по классам

WordCloud визуализирует наиболее частотные слова каждого класса на сырых текстах
(до предобработки). Это даёт интуитивное представление о лексиконе классов.


In [ ]:
plot_wordclouds_by_class(df)


**Вывод по облакам слов (заполнить после запуска).**

- Для positive чаще всего видны слова: `...`.
- Для negative чаще всего видны слова: `...`.
- Для neutral, если класс есть, чаще всего видны слова: `...`.
- Шумные или слишком общие слова, которые мешают интерпретации: `...`.
- Есть ли визуально различимый лексический сигнал тональности: `...`.


### 3.3 Топ-20 слов по классам

Гистограмма частот 20 самых популярных слов для каждого класса
после грубой фильтрации служебных частей речи через `Counter`.
Это количественная картина поверх облаков слов.

> Слово «не» **намеренно оставлено** в обоих наборах —
> оно несёт семантическую нагрузку в негативных отзывах.


In [ ]:
plot_top_words_by_class(df)


**Вывод по топ-20 словам (заполнить после запуска).**

- Топ-слова positive: `...`.
- Топ-слова negative: `...`.
- Топ-слова neutral, если класс есть: `...`.
- Совпадает ли количественный топ с облаками слов: `...`.
- Как ведёт себя частица «не» и почему её важно сохранить: `...`.


### 3.4 Общая статистика словаря

Считаем суммарный объём словаря, среднюю и медианную длину отзыва.
Эта числовая сводка дополняет визуализации.


In [ ]:
all_words = [
    w.lower() for text in df["review"].astype(str)
    for w in re.findall(r"[а-яёА-ЯЁ]+", text)
]
print(f"Всего токенов:      {len(all_words):>10,}")
print(f"Уникальных токенов: {len(set(all_words)):>10,}")
print(f"\nСредняя длина отзыва: {df['word_count'].mean():.1f} слов")
print(f"Медиана длины:        {df['word_count'].median():.1f} слов")
print(f"Максимальная длина:   {df['word_count'].max()} слов")


**Вывод по статистике словаря (заполнить после запуска).**

- Всего токенов: `...`; уникальных токенов: `...`.
- Средняя / медианная / максимальная длина отзыва: `...` / `...` / `...`.
- Достаточно ли ограничения `max_features=50_000` для baseline TF-IDF: `...`.
- Есть ли признаки слишком длинных отзывов или сильного шума в словаре: `...`.


## Предобработка текстов

Предобработка приводит тексты к нормализованному виду:
нижний регистр → удаление пунктуации и цифр → лемматизация spaCy →
удаление стоп-слов (кроме «не» и «нет»).
Результат сохраняется в столбце `cleaned_text`.


### 4.1 Набор стоп-слов

Используем встроенный список spaCy для русского языка,
но исключаем «не» и «нет» — они принципиально меняют смысл высказывания
(ср. «хороший» vs «не хороший»).


In [ ]:
STOPWORDS = set(SPACY_RU_STOPWORDS)
STOPWORDS.discard("не")
STOPWORDS.discard("нет")

print(f"Стоп-слов в наборе: {len(STOPWORDS)}")
print(f"'не' в стоп-словах: {'не' in STOPWORDS}")
print(f"'нет' в стоп-словах: {'нет' in STOPWORDS}")


### 4.2 Функция предобработки одного текста

Шаги:
1. Нижний регистр.
2. Удаление пунктуации и цифр (оставляем только кириллицу и пробелы).
3. Лемматизация через spaCy: каждый токен заменяется нормальной формой.
4. Удаление стоп-слов и коротких токенов (len < 2).


In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яёa-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_doc(doc) -> str:
    return " ".join(
        token.lemma_ for token in doc
        if token.lemma_ not in STOPWORDS
        and len(token.lemma_) >= 2
        and not token.is_space
    )


def preprocess_text(text: str) -> str:
    return clean_doc(nlp(normalize_text(text)))


def preprocess_texts(texts, batch_size: int = 256, n_process: int = 1) -> list[str]:
    normalized_texts = [normalize_text(text) for text in texts]
    docs = nlp.pipe(normalized_texts, batch_size=batch_size, n_process=n_process)
    return [
        clean_doc(doc)
        for doc in tqdm(
            docs,
            total=len(normalized_texts),
            desc="Preprocessing",
            dynamic_ncols=True,
            mininterval=1.0,
            leave=False,
        )
    ]


# Smoke test
sample = "Этот фильм не оставил меня равнодушным! Просто великолепно."
print("Вход: ", sample)
print("Выход:", preprocess_text(sample))

**Вывод по smoke-test предобработки (заполнить после запуска).**

- Сохранилась ли частица «не»: `...`.
- Убрались ли пунктуация, цифры и лишние символы: `...`.
- Видна ли лемматизация русских слов: `...`.
- Что нужно проверить на полном датасете перед обучением: `...`.


### 4.3 Применение к датасету

Обрабатываем весь датасет батчами через `nlp.pipe`.
Если рядом с исходным датасетом уже есть валидный кэш с `cleaned_text`,
ячейка загружает его и пропускает долгий проход spaCy.
Новый кэш сохраняется в Parquet, а параметры предобработки — в JSON рядом с ним.


In [ ]:
PREPROCESS_BATCH_SIZE = 256
PREPROCESS_N_PROCESS = min(4, os.cpu_count() or 1)
PREPROCESSING_VERSION = "spacy-lemma-v1"

CLEANED_PARQUET_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.parquet")
CLEANED_CSV_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.csv")
CACHE_META_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.meta.json")
CACHE_META_KEYS = (
    "spacy_model",
    "spacy_model_version",
    "preprocessing_version",
    "source_rows",
    "source_size_bytes",
    "review_chars_total",
    "stopwords",
)


def build_cache_metadata(source_df: pd.DataFrame) -> dict:
    return {
        "spacy_model": SPACY_MODEL,
        "spacy_model_version": nlp.meta.get("version"),
        "preprocessing_version": PREPROCESSING_VERSION,
        "source_rows": int(len(source_df)),
        "source_path": str(DATA_PATH.resolve()),
        "source_size_bytes": int(DATA_PATH.stat().st_size) if DATA_PATH.exists() else None,
        "review_chars_total": int(source_df["review"].fillna("").str.len().sum()),
        "stopwords": sorted(STOPWORDS),
    }


def cache_frame_is_usable(cached_df: pd.DataFrame, source_df: pd.DataFrame) -> bool:
    if "cleaned_text" not in cached_df.columns:
        return False
    if len(cached_df) != len(source_df):
        return False

    required_columns = [column for column in ("review", "sentiment") if column in source_df.columns]
    return all(column in cached_df.columns for column in required_columns)


def cache_metadata_matches(expected_meta: dict) -> bool:
    if not CACHE_META_PATH.exists():
        return False

    try:
        with CACHE_META_PATH.open("r", encoding="utf-8") as cache_meta_file:
            actual_meta = json.load(cache_meta_file)
    except (OSError, json.JSONDecodeError):
        return False

    return all(actual_meta.get(key) == expected_meta.get(key) for key in CACHE_META_KEYS)


def write_cache_metadata(meta: dict) -> None:
    with CACHE_META_PATH.open("w", encoding="utf-8") as cache_meta_file:
        json.dump(meta, cache_meta_file, ensure_ascii=False, indent=2)


def read_cleaned_cache(cache_path: Path) -> pd.DataFrame:
    if cache_path.suffix == ".parquet":
        return pd.read_parquet(cache_path)
    return pd.read_csv(cache_path)


def find_cleaned_cache(source_df: pd.DataFrame, expected_meta: dict):
    for cache_path in (CLEANED_PARQUET_PATH, CLEANED_CSV_PATH):
        if not cache_path.exists():
            continue

        print(f"Found cleaned cache: {cache_path.resolve()}")
        cached_df = read_cleaned_cache(cache_path)
        if not cache_frame_is_usable(cached_df, source_df):
            print("Cache schema or row count does not match. Rebuilding...")
            continue

        if cache_metadata_matches(expected_meta):
            return cached_df, cache_path

        if not CACHE_META_PATH.exists():
            print("Cache metadata is missing; using cache after basic shape check.")
            write_cache_metadata(expected_meta)
            return cached_df, cache_path

        print("Cache metadata does not match current preprocessing. Rebuilding...")

    return None, None


expected_cache_meta = build_cache_metadata(df)
cached_df, cache_path = find_cleaned_cache(df, expected_cache_meta)

if cached_df is not None:
    df = cached_df
    print(f"Loaded cleaned data from cache → {cache_path.resolve()}")
    if cache_path != CLEANED_PARQUET_PATH:
        df.to_parquet(CLEANED_PARQUET_PATH, index=False)
        write_cache_metadata(expected_cache_meta)
        print(f"Parquet cache saved → {CLEANED_PARQUET_PATH.resolve()}")
else:
    df["cleaned_text"] = preprocess_texts(
        df["review"].fillna("").tolist(),
        batch_size=PREPROCESS_BATCH_SIZE,
        n_process=PREPROCESS_N_PROCESS,
    )
    df.to_parquet(CLEANED_PARQUET_PATH, index=False)
    write_cache_metadata(expected_cache_meta)
    print(f"Cleaned data saved → {CLEANED_PARQUET_PATH.resolve()}")

print("\nПримеры предобработки:")
for _, row in df.sample(3, random_state=SEED).iterrows():
    print("ORIGINAL:", str(row["review"])[:120])
    print("CLEANED: ", str(row["cleaned_text"])[:120])
    print()


### 4.4 Проверка качества предобработки

Проверяем, что предобработка не породила пустых строк,
и смотрим на по одному примеру каждого класса.


In [ ]:
empty_mask = df["cleaned_text"].str.strip().str.len() == 0
print(f"Пустых cleaned_text: {empty_mask.sum()}")
df = df[~empty_mask].reset_index(drop=True)

for cls in df["sentiment"].unique():
    print(f"\n--- {cls.upper()} ---")
    row = df[df["sentiment"] == cls].sample(1, random_state=SEED).iloc[0]
    print("ORIGINAL:", str(row["review"])[:150])
    print("CLEANED: ", str(row["cleaned_text"])[:150])


**Вывод по качеству предобработки на датасете (заполнить после запуска).**

- Количество пустых `cleaned_text`: `...`.
- Насколько читаемыми остались очищенные тексты по примерам классов: `...`.
- Есть ли признаки агрессивной очистки или потери важных слов: `...`.
- Нужно ли менять стоп-слова, регулярное выражение или параметры `nlp.pipe`: `...`.


## Подготовка данных к обучению

Перед обучением моделей выполняем два обязательных шага:
стратифицированное разделение на обучающую и тестовую выборки,
а затем TF-IDF векторизацию очищенных текстов.
Базовые модели используют общий TF-IDF для честного нулевого сравнения,
а Optuna-модели получают собственный `TfidfVectorizer` внутри `Pipeline`,
чтобы словарь и IDF обучались заново на каждом CV-фолде.


### 5.1 Train / test split

Стратифицированное разделение 80 / 20 с фиксированным `SEED`
обеспечивает одинаковое соотношение классов в обеих частях.


In [ ]:
X = df["cleaned_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {len(X_train)},  Test: {len(X_test)}")
print("\nТренировочное распределение классов:")
print(y_train.value_counts(normalize=True).round(3))


### 5.2 TF-IDF векторизация

`TfidfVectorizer` с параметрами, рекомендованными для задач классификации текста:
- `sublinear_tf=True` — логарифмическое масштабирование TF снижает влияние очень частых слов;
- `ngram_range=(1, 2)` — учёт биграмм захватывает контекст «не хороший»;
- `max_features=50 000` — ограничение размерности для управляемости;
- `min_df=2` — игнорируем слова, встречающиеся только один раз (шум).


In [ ]:
TFIDF_BASE_PARAMS = {
    "sublinear_tf": True,
    "ngram_range": (1, 2),
    "max_features": 50_000,
    "min_df": 2,
    "analyzer": "word",
}

tfidf = TfidfVectorizer(**TFIDF_BASE_PARAMS)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("Train matrix:", X_train_tfidf.shape)
print("Test  matrix:", X_test_tfidf.shape)
print(f"Словарь TF-IDF: {len(tfidf.vocabulary_):,} признаков")

**Вывод по TF-IDF матрицам (заполнить после запуска).**

- Размер train-матрицы: `...`; размер test-матрицы: `...`.
- Размер словаря TF-IDF: `...` признаков.
- Достаточно ли признаков для baseline-моделей, нет ли чрезмерной размерности: `...`.
- Почему для Optuna ниже векторизатор остаётся внутри `Pipeline`: `...`.


## Классификация

Основной раздел работы. Сравниваем три обязательные baseline-модели:
LogisticRegression, LinearSVC и CatBoostClassifier.
Затем для каждой из этих трёх моделей подбираем гиперпараметры через Optuna,
добавляем TF-IDF + LSA pipeline как отдельную конфигурацию
и выбираем лучшую модель по macro F1.

Режим обучения CatBoost берётся из `DEVICE_MODE`: на CUDA-машине можно включить
GPU, а на macOS или другом окружении без CUDA ноутбук остаётся на CPU.


### 6.1 Базовые модели (нулевой проход)

Обучаем три обязательные модели на общей TF-IDF матрице с фиксированными стартовыми параметрами.
Это даёт baseline перед Optuna-подбором и показывает, есть ли прирост после тюнинга.
CatBoost использует `CATBOOST_DEVICE_PARAMS`, поэтому один и тот же код работает на CPU и CUDA.


In [ ]:
import time

catboost_loss = "MultiClass" if y_train.nunique() > 2 else "Logloss"

MODELS_BASELINE = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=SEED
    ),
    "LinearSVC": LinearSVC(
        max_iter=3000, class_weight="balanced", random_state=SEED
    ),
    "CatBoostClassifier": CatBoostClassifier(
        **CATBOOST_DEVICE_PARAMS,
        iterations=500,
        learning_rate=0.1,
        depth=6,
        loss_function=catboost_loss,
        auto_class_weights="Balanced",
        random_seed=SEED,
        verbose=0,
        thread_count=-1,
        allow_writing_files=False,
    ),
}

baseline_results = {}
for name, model in MODELS_BASELINE.items():
    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    preds = np.array(model.predict(X_test_tfidf)).flatten()
    elapsed = time.time() - t0
    f1 = f1_score(y_test, preds, average="macro")
    acc = accuracy_score(y_test, preds)
    baseline_results[name] = {
        "macro_f1": round(f1, 4),
        "accuracy": round(acc, 4),
        "time_s": round(elapsed, 2),
    }
    print(f"{name:<25} macro F1={f1:.4f}  acc={acc:.4f}  ({elapsed:.2f}s)")

baseline_df = pd.DataFrame(baseline_results).T.sort_values("macro_f1", ascending=False)
display(baseline_df)


**Вывод по baseline-моделям (заполнить после запуска).**

- Лучшая baseline-модель по `macro_f1`: `...` (`macro_f1=...`, `accuracy=...`).
- Самая быстрая baseline-модель: `...`; самая медленная: `...`.
- Как CatBoost выглядит относительно LogisticRegression и LinearSVC по качеству и времени: `...`.
- Есть ли разрыв между `accuracy` и `macro_f1`, который указывает на влияние дисбаланса: `...`.


### 6.2 Optuna — пространства поиска

Для улучшения используем Optuna: она подбирает параметры векторизации и классификатора
по `f1_macro` на стратифицированной кросс-валидации.
`TfidfVectorizer` находится внутри `Pipeline`, поэтому каждый trial честно обучает словарь
и IDF только на обучающей части соответствующего CV-фолда.


In [ ]:
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
N_OPTUNA_TRIALS = 20
OPTUNA_TIMEOUT_PER_MODEL = None  # можно задать число секунд, если нужно ограничить время


def make_tfidf(params: dict) -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="word",
        min_df=params["tfidf_min_df"],
        max_features=params["tfidf_max_features"],
        ngram_range=(1, params["tfidf_ngram_max"]),
        sublinear_tf=params["tfidf_sublinear_tf"],
    )


def suggest_tfidf_params(trial, *, max_features_choices=(30_000, 50_000, 80_000)) -> dict:
    return {
        "tfidf_min_df": trial.suggest_int("tfidf_min_df", 1, 5),
        "tfidf_max_features": trial.suggest_categorical("tfidf_max_features", list(max_features_choices)),
        "tfidf_ngram_max": trial.suggest_categorical("tfidf_ngram_max", [1, 2]),
        "tfidf_sublinear_tf": trial.suggest_categorical("tfidf_sublinear_tf", [True, False]),
    }


def suggest_lr_params(trial) -> dict:
    params = suggest_tfidf_params(trial)
    params.update({
        "clf_C": trial.suggest_float("clf_C", 0.05, 10.0, log=True),
    })
    return params


def build_lr_pipeline(params: dict) -> Pipeline:
    return Pipeline([
        ("tfidf", make_tfidf(params)),
        ("clf", LogisticRegression(
            C=params["clf_C"],
            max_iter=1000,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])


def suggest_svc_params(trial) -> dict:
    params = suggest_tfidf_params(trial)
    params.update({
        "clf_C": trial.suggest_float("clf_C", 0.05, 10.0, log=True),
        "clf_tol": trial.suggest_float("clf_tol", 1e-5, 1e-3, log=True),
    })
    return params


def build_svc_pipeline(params: dict) -> Pipeline:
    return Pipeline([
        ("tfidf", make_tfidf(params)),
        ("clf", LinearSVC(
            C=params["clf_C"],
            tol=params["clf_tol"],
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])


def suggest_catboost_params(trial) -> dict:
    params = suggest_tfidf_params(trial, max_features_choices=(5_000, 10_000, 20_000))
    params.update({
        "clf_iterations": trial.suggest_int("clf_iterations", 200, 800, step=100),
        "clf_depth": trial.suggest_int("clf_depth", 4, 8),
        "clf_learning_rate": trial.suggest_float("clf_learning_rate", 0.03, 0.3, log=True),
        "clf_l2_leaf_reg": trial.suggest_float("clf_l2_leaf_reg", 1.0, 10.0, log=True),
    })
    return params


def build_catboost_pipeline(params: dict) -> Pipeline:
    return Pipeline([
        ("tfidf", make_tfidf(params)),
        ("clf", CatBoostClassifier(
            **CATBOOST_DEVICE_PARAMS,
            iterations=params["clf_iterations"],
            depth=params["clf_depth"],
            learning_rate=params["clf_learning_rate"],
            l2_leaf_reg=params["clf_l2_leaf_reg"],
            loss_function=catboost_loss,
            auto_class_weights="Balanced",
            random_seed=SEED,
            verbose=0,
            thread_count=-1,
            allow_writing_files=False,
        )),
    ])


def optimize_with_optuna(name: str, suggest_params, build_pipeline, *, cv_n_jobs: int = -1) -> dict:
    def objective(trial):
        params = suggest_params(trial)
        pipeline = build_pipeline(params)
        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=CV,
            scoring="f1_macro",
            n_jobs=cv_n_jobs,
        )
        trial.set_user_attr("cv_std", float(scores.std()))
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    started_at = time.time()
    with tqdm(
        total=N_OPTUNA_TRIALS,
        desc=f"Optuna: {name}",
        dynamic_ncols=True,
        mininterval=1.0,
        leave=False,
    ) as progress:
        for _ in range(N_OPTUNA_TRIALS):
            remaining_time = None
            if OPTUNA_TIMEOUT_PER_MODEL is not None:
                remaining_time = max(0.0, OPTUNA_TIMEOUT_PER_MODEL - (time.time() - started_at))
                if remaining_time == 0.0:
                    break

            completed_before = len(study.trials)
            study.optimize(
                objective,
                n_trials=1,
                timeout=remaining_time,
                show_progress_bar=False,
            )
            completed_delta = len(study.trials) - completed_before
            if completed_delta == 0:
                break

            progress.update(completed_delta)
            try:
                progress.set_postfix(best=f"{study.best_value:.4f}")
            except ValueError:
                pass

    best_model = build_pipeline(study.best_params)
    t0 = time.time()
    best_model.fit(X_train, y_train)
    elapsed = time.time() - t0

    preds = np.array(best_model.predict(X_test)).flatten()
    f1 = f1_score(y_test, preds, average="macro")
    acc = accuracy_score(y_test, preds)

    print(f"\n{name}")
    print("Best CV macro F1:", round(study.best_value, 4))
    print("Best params:", study.best_params)
    print(f"Test macro F1={f1:.4f}  acc={acc:.4f}  fit_time={elapsed:.2f}s")
    print("\n", classification_report(y_test, preds))

    return {
        "study": study,
        "model": best_model,
        "preds": preds,
        "macro_f1": f1,
        "accuracy": acc,
        "fit_time_s": elapsed,
    }


### 6.3 Optuna — запуск оптимизации трёх моделей

Запускаем одинаковую процедуру для LogisticRegression, LinearSVC и CatBoostClassifier.
Для CatBoost ограничиваем размер TF-IDF-словаря меньшими значениями: иначе перебор становится слишком долгим
на разреженных текстовых признаках. CatBoost-trial'ы запускаются последовательно (`cv_n_jobs=1`),
чтобы CPU/GPU-ресурс не дробился между фолдами.


In [ ]:
lr_run = optimize_with_optuna(
    "LogisticRegression",
    suggest_lr_params,
    build_lr_pipeline,
    cv_n_jobs=-1,
)
svc_run = optimize_with_optuna(
    "LinearSVC",
    suggest_svc_params,
    build_svc_pipeline,
    cv_n_jobs=-1,
)
catboost_run = optimize_with_optuna(
    "CatBoostClassifier",
    suggest_catboost_params,
    build_catboost_pipeline,
    cv_n_jobs=1,
)

best_lr = lr_run["model"]
lr_preds = lr_run["preds"]
lr_f1 = lr_run["macro_f1"]
lr_acc = lr_run["accuracy"]

best_svc = svc_run["model"]
svc_preds = svc_run["preds"]
svc_f1 = svc_run["macro_f1"]
svc_acc = svc_run["accuracy"]

best_catboost = catboost_run["model"]
catboost_optuna_preds = catboost_run["preds"]
catboost_f1 = catboost_run["macro_f1"]
catboost_acc = catboost_run["accuracy"]

optuna_summary_df = pd.DataFrame({
    "LR tuned": {
        "cv_macro_f1": lr_run["study"].best_value,
        "test_macro_f1": lr_f1,
        "test_accuracy": lr_acc,
    },
    "LinearSVC tuned": {
        "cv_macro_f1": svc_run["study"].best_value,
        "test_macro_f1": svc_f1,
        "test_accuracy": svc_acc,
    },
    "CatBoost tuned": {
        "cv_macro_f1": catboost_run["study"].best_value,
        "test_macro_f1": catboost_f1,
        "test_accuracy": catboost_acc,
    },
}).T.sort_values("test_macro_f1", ascending=False)

display(optuna_summary_df.round(4))


**Вывод по Optuna-подбору (заполнить после запуска).**

- Лучшая tuned-модель по `test_macro_f1`: `...`.
- Прирост LogisticRegression относительно baseline: `...`.
- Прирост LinearSVC относительно baseline: `...`.
- Прирост CatBoost относительно baseline: `...`.
- Лучшие параметры TF-IDF и классификатора, которые стоит упомянуть в отчёте: `...`.
- Есть ли заметный разрыв между CV-метрикой и тестовой метрикой: `...`.


### 6.4 TF-IDF + LSA (TruncatedSVD)

Латентно-семантический анализ (LSA) сжимает TF-IDF матрицу до 300 скрытых тем,
захватывая синонимию и тематическую близость слов.
Сравниваем с прямым TF-IDF подходом как отдельную дополнительную конфигурацию.


In [ ]:
lsa_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(**TFIDF_BASE_PARAMS)),
    ("svd",   TruncatedSVD(n_components=300, random_state=SEED)),
    ("clf",   LogisticRegression(
        C=5.0, max_iter=1000, class_weight="balanced", random_state=SEED
    )),
])

lsa_pipeline.fit(X_train, y_train)
lsa_preds = lsa_pipeline.predict(X_test)
lsa_f1  = f1_score(y_test, lsa_preds, average="macro")
lsa_acc = accuracy_score(y_test, lsa_preds)
print(f"LSA Pipeline  macro F1={lsa_f1:.4f}  acc={lsa_acc:.4f}")
print("\n", classification_report(y_test, lsa_preds))

**Вывод по TF-IDF + LSA (заполнить после запуска).**

- `macro_f1` LSA pipeline: `...`; `accuracy`: `...`.
- LSA лучше или хуже лучшей прямой TF-IDF-модели: `...`.
- Какие классы по `classification_report` просели сильнее всего: `...`.
- Стоит ли оставлять LSA как рабочий вариант или только как сравнение: `...`.


### 6.5 Сводная таблица результатов

Сравниваем baseline, Optuna-tuned модели и TF-IDF + LSA по accuracy и macro F1
на тестовой выборке. Лучшая модель дальше используется для анализа ошибок
и проверки на новых текстах.


In [ ]:
def get_scores(model, X, y_true):
    preds = np.array(model.predict(X)).flatten()
    return {
        "macro_f1": round(f1_score(y_true, preds, average="macro"), 4),
        "accuracy": round(accuracy_score(y_true, preds), 4),
    }

results = {
    "LR baseline": get_scores(MODELS_BASELINE["LogisticRegression"], X_test_tfidf, y_test),
    "LinearSVC baseline": get_scores(MODELS_BASELINE["LinearSVC"], X_test_tfidf, y_test),
    "CatBoost baseline": get_scores(MODELS_BASELINE["CatBoostClassifier"], X_test_tfidf, y_test),
    "LR tuned": {"macro_f1": lr_f1, "accuracy": lr_acc},
    "LinearSVC tuned": {"macro_f1": svc_f1, "accuracy": svc_acc},
    "CatBoost tuned": {"macro_f1": catboost_f1, "accuracy": catboost_acc},
    "TF-IDF + LSA": {"macro_f1": lsa_f1, "accuracy": lsa_acc},
}

results_df = (
    pd.DataFrame(results).T
    .assign(macro_f1=lambda d: d["macro_f1"].astype(float),
            accuracy=lambda d: d["accuracy"].astype(float))
    .sort_values("macro_f1", ascending=False)
)
display(results_df)

best_model_name = results_df.index[0]
print(f"\nЛучшая модель: {best_model_name}")


**Вывод по сводной таблице результатов (заполнить после запуска).**

- Итоговая лучшая модель: `...`.
- Лучшие значения `macro_f1` и `accuracy`: `...` / `...`.
- Разница между лучшей tuned-моделью и лучшим baseline: `...`.
- Какой вариант оптимален не только по качеству, но и по скорости/простоте запуска: `...`.


### 6.6 Матрица ошибок лучшей модели

Матрица ошибок показывает абсолютные числа правильных и неправильных
классификаций для каждой пары (истинный класс, предсказанный класс).

In [ ]:
preds_map = {
    "LR baseline": MODELS_BASELINE["LogisticRegression"].predict(X_test_tfidf),
    "LinearSVC baseline": MODELS_BASELINE["LinearSVC"].predict(X_test_tfidf),
    "CatBoost baseline": np.array(MODELS_BASELINE["CatBoostClassifier"].predict(X_test_tfidf)).flatten(),
    "LR tuned": lr_preds,
    "LinearSVC tuned": svc_preds,
    "CatBoost tuned": catboost_optuna_preds,
    "TF-IDF + LSA": lsa_preds,
}

best_preds = preds_map[best_model_name]
labels_sorted = sorted(y_test.unique())

plot_confusion_matrix_plotly(
    y_test,
    best_preds,
    labels_sorted,
    title=f"Матрица ошибок — {best_model_name}",
)


**Вывод по матрице ошибок лучшей модели (заполнить после запуска).**

- Больше всего модель путает `...` с `...`.
- Лучше всего распознаётся класс `...`, хуже всего — `...`.
- Согласуется ли матрица ошибок с выбранной метрикой `macro_f1`: `...`.
- Какие ошибки наиболее критичны для задачи анализа тональности: `...`.


## Анализ ошибок

Анализируем, какие конкретные тексты модель классифицировала неверно.
Это помогает выявить систематические слабости:
саркастические тексты, смешанные отзывы, очень короткие фрагменты.


### 7.1 Сбор ошибочных предсказаний

Собираем все тексты, где предсказание расходится с истинной меткой,
и показываем образцы ложноположительных и ложноотрицательных классификаций.

In [ ]:
test_df = X_test.reset_index(drop=True).to_frame("cleaned_text")
test_df["original"]   = df.loc[X_test.index, "review"].values
test_df["true_label"] = y_test.reset_index(drop=True)
test_df["pred_label"] = best_preds

errors = test_df[test_df["true_label"] != test_df["pred_label"]].copy()
print(f"Всего ошибок: {len(errors)} из {len(test_df)} ({100*len(errors)/len(test_df):.1f}%)")
print(f"Правильно:    {len(test_df) - len(errors)} ({100*(1 - len(errors)/len(test_df)):.1f}%)")

# Show unique error combinations
print("\nРаспределение ошибок по типам:")
print(errors.groupby(["true_label", "pred_label"]).size().rename("count"))


In [ ]:
# Show samples for each error type
for (true_l, pred_l), group in errors.groupby(["true_label", "pred_label"]):
    print(f"\n=== Истинный: {true_l.upper()} → Предсказан: {pred_l.upper()} ===")
    for _, row in group.head(3).iterrows():
        print(f"  [{row['true_label']}→{row['pred_label']}] {str(row['original'])[:160]}")


### 7.2 Характер ошибок

**Вывод по ошибочным предсказаниям (заполнить после запуска).**

- Доля ошибок на тестовой выборке: `...`% (`...` из `...`).
- Основные направления ошибок по парам `true_label → pred_label`: `...`.
- Типичные свойства ошибочных отзывов: сарказм / смешанная оценка / короткий текст / редкая лексика / другое: `...`.
- Примеры, которые лучше всего иллюстрируют проблему модели: `...`.
- Что это говорит об ограничениях TF-IDF и выбранной модели: `...`.


## Тест на новых текстах

Проверяем лучшую модель на пяти новых отзывах в принципиально разных стилях,
пропуская их через ту же самую функцию `preprocess_text`.
Тексты написаны намеренно в разных регистрах, чтобы не подыгрывать модели.


### 8.1 Пять тестовых текстов

Пять отзывов покрывают разные стилистические регистры:
разговорный слэнг, формальный академический, сарказм, нейтральный фактический
и эмоционально насыщенный негатив.

In [ ]:
test_texts = [
    # 1. Short enthusiastic slang — expected: positive
    "Топчик! Смотрел 3 раза подряд, реально крутой фильм, всем советую!",

    # 2. Formal academic-style negative — expected: negative
    "Данное кинопроизведение не отвечает минимальным требованиям качественного "
    "повествования. Режиссёрская концепция лишена внутренней логики, "
    "а актёрская игра не выдерживает никакой критики.",

    # 3. Sarcastic mixed review — expected: negative (hard case)
    "Гениальная картина, если вы любите тратить 2 часа жизни впустую. "
    "Восхитительно бессмысленный сюжет и незабываемо слабая игра актёров.",

    # 4. Very brief neutral/factual — expected: neutral if the class exists, ambiguous otherwise
    "Фильм 2019 года. Режиссёр Иванов. Посмотрел. Актёры играют нормально.",

    # 5. Emotional negative — expected: negative
    "Просто ужасно!! Деньги на ветер! Никогда не смотрите этот мусор! "
    "Хуже фильма в жизни не видел, полный провал.",
]

available_labels = set(y_train.unique())
labels_expected = [
    "positive",
    "negative",
    "negative",
    "neutral" if "neutral" in available_labels else "ambiguous",
    "negative",
]

for i, (text, expected) in enumerate(zip(test_texts, labels_expected), 1):
    print(f"[{i}] Ожидается: {expected}")
    print(f"    Текст: {text[:90]}...")
    print()

### 8.2 Предобработка и предсказание

Пропускаем каждый текст через `preprocess_texts` —
ту же функцию, что применялась к обучающим данным.
Baseline-модели получают общий обученный `tfidf`, а tuned/LSA-модели получают
очищенный текст напрямую, потому что их векторизатор находится внутри `Pipeline`.
Для моделей с `predict_proba` показываем вероятность, для LinearSVC — margin
(`decision_function`), а не вероятность.


In [ ]:
def predict_with_score(model, X):
    predictions = np.array(model.predict(X)).flatten()
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X).max(axis=1)
        score_type = "probability"
    elif hasattr(model, "decision_function"):
        raw_scores = model.decision_function(X)
        scores = np.abs(raw_scores) if raw_scores.ndim == 1 else raw_scores.max(axis=1)
        score_type = "margin"
    else:
        scores = np.full(len(predictions), np.nan)
        score_type = "n/a"
    return predictions, scores, score_type


model_obj_map = {
    "LR baseline": (MODELS_BASELINE["LogisticRegression"], "vectorized"),
    "LinearSVC baseline": (MODELS_BASELINE["LinearSVC"], "vectorized"),
    "CatBoost baseline": (MODELS_BASELINE["CatBoostClassifier"], "vectorized"),
    "LR tuned": (best_lr, "text"),
    "LinearSVC tuned": (best_svc, "text"),
    "CatBoost tuned": (best_catboost, "text"),
    "TF-IDF + LSA": (lsa_pipeline, "text"),
}

best_model_obj, input_kind = model_obj_map[best_model_name]

# Preprocess new texts with the same function used for train data
cleaned_new = preprocess_texts(test_texts, batch_size=PREPROCESS_BATCH_SIZE, n_process=PREPROCESS_N_PROCESS)

if input_kind == "text":
    predictions_new, score_values, score_type = predict_with_score(best_model_obj, cleaned_new)
else:
    new_vectors = tfidf.transform(cleaned_new)
    predictions_new, score_values, score_type = predict_with_score(best_model_obj, new_vectors)

results_new = pd.DataFrame({
    "Текст": [text[:55] + "..." for text in test_texts],
    "Ожидается": labels_expected,
    "Предсказано": predictions_new,
    "Скор модели": np.round(score_values, 3),
    "Тип скора": score_type,
})
display(results_new)


In [ ]:
print("Предобработанные тексты:")
for i, (orig, cleaned) in enumerate(zip(test_texts, cleaned_new), 1):
    print(f"[{i}] ORIGINAL: {orig[:80]}")
    print(f"    CLEANED:  {cleaned[:80]}")
    print()


### 8.3 Комментарий к предсказаниям

**Комментарии к пяти новым текстам (заполнить после запуска).**

- Текст 1, разговорный позитив: ожидалось `...`, предсказано `...`, скор `...`; комментарий: `...`.
- Текст 2, формальный негатив: ожидалось `...`, предсказано `...`, скор `...`; комментарий: `...`.
- Текст 3, сарказм: ожидалось `...`, предсказано `...`, скор `...`; комментарий: `...`.
- Текст 4, короткий нейтральный/пограничный: ожидалось `...`, предсказано `...`, скор `...`; комментарий: `...`.
- Текст 5, эмоциональный негатив: ожидалось `...`, предсказано `...`, скор `...`; комментарий: `...`.


**Общий вывод по тесту на новых текстах (заполнить после запуска).**

- На каких стилях модель сработала уверенно: `...`.
- Где модель ошиблась или дала низкий скор: `...`.
- Подтверждаются ли наблюдения из анализа ошибок раздела 7: `...`.
- Что этот мини-тест показывает о применимости модели к реальным отзывам: `...`.


## Выводы

**Итоговый вывод (заполнить после полного запуска ноутбука).**

1. Данные и предобработка: размер датасета, классы, качество очистки, сохранение «не»: `...`.
2. EDA: что показали Plotly-графики по дисбалансу, длинам и лексике классов: `...`.
3. Baseline: какая из трёх моделей стала лучшей до тюнинга и почему: `...`.
4. Optuna: дала ли оптимизация прирост относительно baseline, какие параметры оказались удачными: `...`.
5. CatBoost: был ли полезен как обязательная третья модель, каков компромисс качество/время: `...`.
6. LSA: улучшило ли сжатие TF-IDF результат или ухудшило его: `...`.
7. Лучшая модель: название, `macro_f1`, `accuracy`, главные ошибки по confusion matrix: `...`.
8. Новые тексты: какие кейсы модель распознала правильно, где проявились ограничения: `...`.
9. Дальнейшие улучшения: что стоит попробовать после этой версии: `...`.
